In [10]:
# These are the libraries you may use for this assignment.
# If you would like to use any other libraries, email james@uwaterloo.ca for permission first.

from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

## CFM 101 — Group Assignment 2026
### Robo-Advising Challenge: Two Clients, Two Strategies

**Team Number:** XX  
**Team Member Names (three or four):**  

Enter the names of all participating members below (one name per line):

**Submission:** Friday, November 27, 2026, at 9:00 AM (Eastern), via Dropbox.  
**Submit this completed Jupyter notebook only.**

Your team must construct **both** portfolios from the supplied `Tickers.csv` file:

- **Client A — Growth:** $\$1,000,000$ CAD; seek the highest expected return.
- **Client B — Low Risk:** $\$750,000$ CAD; seek the lowest portfolio volatility.

Both clients require 10–20 eligible stocks, at most 15% in any stock and at most 40% in any sector. Read the full assignment handout for all eligibility, fee, currency, and competition rules. Do not hardcode the selected stocks. Check that tickers are valid. The supplied ticker universe will permit feasible portfolios.

## Data preparation and shared functions

Read `Tickers.csv`, check ticker eligibility, retrieve the data needed for your strategies, and create any functions you will reuse for both clients. Use closing prices from **November 27, 2026** for the purchase calculations. All portfolio values and weights must be calculated in CAD, with purchase fees paid from the client budget.

In [11]:
TEAM = "XX"
budget = 750000
buy_date = pd.Timestamp("2026-11-27")
start = "2025-10-01"
end = "2026-09-30"
fx_symbol = "USDCAD=X"

# The TA file takes priority. The example has no header and includes a duplicate.
try:
    ticker_file = "Tickers.csv"
    ticker_column = pd.read_csv(ticker_file, header=None, dtype=str)[0]
except FileNotFoundError:
    ticker_file = "Tickers_Example.csv"
    ticker_column = pd.read_csv(ticker_file, header=None, dtype=str)[0]

tickers = ticker_column.dropna().str.strip().str.upper().drop_duplicates()
tickers = tickers[~tickers.isin(["", "TICKER", "TICKERS", "SYMBOL"])].tolist()
print("Input file:", ticker_file, "| Unique tickers:", len(tickers))

# Use the team's quote type, listing market, currency and sector checks.
rows = []
for ticker in tickers:
    try:
        info = yf.Ticker(ticker).get_info()
    except Exception as error:
        print(ticker, "metadata unavailable:", type(error).__name__)
        info = {}
    rows.append([ticker, info.get("longName"), info.get("quoteType"),
                 info.get("market"), info.get("currency"), info.get("sector")])

all_stocks = pd.DataFrame(rows, columns=["Ticker", "Company", "Type", "Market",
                                         "Currency", "Sector"]).set_index("Ticker")
mask = (all_stocks["Type"].eq("EQUITY")
        & all_stocks["Market"].isin(["us_market", "ca_market"])
        & all_stocks["Currency"].isin(["CAD", "USD"])
        & all_stocks["Sector"].fillna("").str.strip().ne(""))
eligible_stocks = all_stocks.loc[mask].copy()
if eligible_stocks.empty:
    raise ValueError("No usable company profiles. Check the input file and Yahoo Finance.")
display(all_stocks.assign(Profile_pass=mask))

Input file: Tickers_Example.csv | Unique tickers: 40


,Company,Type,Market,Currency,Sector,Profile_pass
Ticker,,,,,,
AAPL,Apple Inc.,EQUITY,us_market,USD,Technology,True
ABBV,AbbVie Inc.,EQUITY,us_market,USD,Healthcare,True
ABT,Abbott Laboratories,EQUITY,us_market,USD,Healthcare,True
ACN,Accenture plc,EQUITY,us_market,USD,Technology,True
AGN,NaN,NONE,NaN,NaN,NaN,False
AIG,"American International Group, Inc.",EQUITY,us_market,USD,Financial Services,True
AMZN,"Amazon.com, Inc.",EQUITY,us_market,USD,Consumer Cyclical,True
AXP,American Express Company,EQUITY,us_market,USD,Financial Services,True
BA,The Boeing Company,EQUITY,us_market,USD,Industrials,True


In [12]:
# Yahoo's end date is exclusive. A single bulk request also supplies historical FX.
today_et = pd.Timestamp.now(tz="America/Toronto")
official = today_et >= pd.Timestamp("2026-11-27 16:00", tz="America/Toronto")
price_date = buy_date if official else pd.Timestamp(end)
print("Official purchase date:" if official else "Practice prices only:", price_date.date())
symbols = eligible_stocks.index.tolist() + [fx_symbol]
history = yf.download(symbols, start=start, end=price_date + pd.Timedelta(days=1),
                      group_by="ticker", auto_adjust=False, progress=False, threads=4)
def get_field(field):
    values = history.xs(field, level=1, axis=1).reindex(columns=symbols)
    values.index = pd.to_datetime(values.index).tz_localize(None).normalize()
    return values.sort_index()

close = get_field("Close")
volume = get_field("Volume")
if price_date not in close.index or pd.isna(close.at[price_date, fx_symbol]):
    raise ValueError("FX close unavailable for the pricing date. Retry when Yahoo publishes it.")
rate = float(close.at[price_date, fx_symbol])
cad_prices = {}
for ticker in eligible_stocks.index:
    traded = volume.loc[start:end, ticker].where(close.loc[start:end, ticker].notna()).dropna()
    counts = traded.groupby(traded.index.to_period("M")).transform("count")
    eligible_stocks.loc[ticker, "Avg_Volume"] = traded[counts >= 18].mean()
    eligible_stocks.loc[ticker, "Close"] = close.at[price_date, ticker]
    prices = close.loc[start:end, ticker]
    factor = 1.0
    if eligible_stocks.at[ticker, "Currency"] == "USD":
        prices = prices.ffill() * close.loc[start:end, fx_symbol].ffill()
        factor = rate
    cad_prices[ticker] = prices
    eligible_stocks.loc[ticker, "Close_CAD"] = close.at[price_date, ticker] * factor

eligible_stocks = eligible_stocks.loc[(eligible_stocks["Avg_Volume"] >= 5000)
                                      & (eligible_stocks["Close_CAD"] > 0)].copy()
# Fixed shares, price returns only. Carry the last close across market holidays.
# Score every candidate on the same dates, starting after all have prices.
stock_days = close.loc[start:end, eligible_stocks.index].notna().any(axis=1)
prices_cad = pd.DataFrame(cad_prices)[eligible_stocks.index].loc[stock_days].ffill().dropna()
daily_returns = prices_cad.pct_change().dropna()
vol = daily_returns.std()
usable = vol[(vol > 0) & np.isfinite(vol)].index
prices_cad = prices_cad[usable]
daily_returns = daily_returns[usable]
vol = vol[usable]
if len(usable) < 10 or len(daily_returns) < 2:
    raise ValueError("Too few usable stocks or common return dates. Inspect the Yahoo data.")
display(eligible_stocks)
print("Usable stocks:", len(usable), "| Common daily returns:", len(daily_returns))

def calc_fee_cad(shares_num):
    return min(0.001 * shares_num, 2.15) * rate

Practice prices only: 2026-09-30


,Company,Type,Market,Currency,Sector,Avg_Volume,Close,Close_CAD
Ticker,,,,,,,,
AAPL,Apple Inc.,EQUITY,us_market,USD,Technology,4.828203e+07,333.019989,472.598646
ABBV,AbbVie Inc.,EQUITY,us_market,USD,Healthcare,6.281745e+06,261.589996,371.230203
ABT,Abbott Laboratories,EQUITY,us_market,USD,Healthcare,1.008423e+07,98.830002,140.252617
ACN,Accenture plc,EQUITY,us_market,USD,Technology,6.116120e+06,183.369995,260.225855
AIG,"American International Group, Inc.",EQUITY,us_market,USD,Financial Services,4.324195e+06,74.419998,105.611650
AMZN,"Amazon.com, Inc.",EQUITY,us_market,USD,Consumer Cyclical,4.652561e+07,249.149994,353.576223
AXP,American Express Company,EQUITY,us_market,USD,Financial Services,3.054224e+06,304.100006,431.557432
BA,The Boeing Company,EQUITY,us_market,USD,Industrials,7.140954e+06,186.050003,264.029135
BAC,Bank of America Corporation,EQUITY,us_market,USD,Financial Services,3.673368e+07,54.430000,77.243245


Usable stocks: 35 | Common daily returns: 255


## Client A — Growth

Construct a portfolio intended to achieve the **highest expected return**. Use your selected stocks and portfolio weights to create a DataFrame named `Portfolio_Final_A` with index **1 through n** and columns (in this order): `Ticker`, `Price`, `Currency`, `Shares`, `Value`, `Weight`. `Price` is in the stock’s quoted currency, `Value` is in CAD, and `Weight` is a fraction summing to 1.

### Client A — Financial justification

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.

## Client B — Low Risk

Construct a portfolio intended to achieve the **lowest risk**, defined as the standard deviation of daily portfolio returns in CAD. Create `Portfolio_Final_B` using the same DataFrame column names, order, and index convention as for Client A.

In [13]:
# Columns are each stock's CAD value after investing one dollar on the first day.
growth = (prices_cad / prices_cad.iloc[0]).values

# Keep sectors and volatility in the same order as the price columns.
names = list(prices_cad.columns)
sectors = list(eligible_stocks.loc[names, "Sector"])
stock_vol = vol.loc[names].values

def path_stats(value):#take out the comparable days.
    #the day to the end and the last second day. so that we can calculate the return of that period
    returns = value[1:] / value[:-1] - 1#we will get a list of returns here.
    total_return = value[-1] / value[0] - 1
    daily_volatility = returns.std(ddof=1)  # Sample standard deviation
    return total_return, daily_volatility

def sector_groups(ids):#divide them into different industries
    groups = {}
    for pos, stock in enumerate(ids):
        sector = sectors[stock]
        if sector not in groups:
            groups[sector] = []
        groups[sector].append(pos)  # Position in the selected portfolio
    return groups

def sector_capacity(groups):
    capacity = 0.0
    for group in groups.values():
        # Each stock is capped at 15% and each sector at 40%.
        sector_max = min(0.15 * len(group), 0.40)
        capacity += sector_max
    return capacity

def capped_weights(scores, caps, total):
    caps = np.array(caps)
    result = np.zeros(len(scores))
    free = caps > 0
    remaining = total
    while free.any() and remaining > 1e-12:
        # Only update positions that are still open.
        proposal = result.copy()
        proposal[free] = scores[free] / scores[free].sum() * remaining
        over = free & (proposal > caps)
        if not over.any():
            return proposal
        # Fix the positions above their caps before the next round.
        result[over] = caps[over]
        free[over] = False
        remaining = total - result.sum()
    return result

def feasible_weights(ids, scores):
    groups = list(sector_groups(ids).values())
    sector_caps = [min(0.40, 0.15 * len(group)) for group in groups]
    if sum(sector_caps) < 1 - 1e-10:
        return None
    sector_scores = np.array([scores[group].sum() for group in groups])
    # Allocate across sectors first, then split each sector's weight among its stocks.
    sector_weights = capped_weights(sector_scores, sector_caps, 1.0)
    weights = np.zeros(len(ids))
    for group, sector_weight in zip(groups, sector_weights):
        weights[group] = capped_weights(scores[group], [0.15] * len(group), sector_weight)
    return weights

def score_final(ids):
    best = None#trying two different methods.
    for label, scores in [
        ("Equal before caps", np.ones(len(ids))),
        ("Inverse volatility before caps", 1 / stock_vol[ids])
    ]:
        weights = feasible_weights(ids, scores)
        if weights is None:
            return None
        portfolio_value = (growth[:, ids] * weights).sum(axis=1)
        risk = path_stats(portfolio_value)[1]
        if best is None or risk < best[0]:
            best = (risk, ids.copy(), weights, label)
    return best

In [14]:
best = None
stock_risks = [(stock_vol[i], i) for i in range(len(names))]
# Try the three lowest-volatility stocks as starting points.
for volatility, seed in sorted(stock_risks)[:3]:
    ids = [seed]
    while len(ids) < min(20, len(names)):
        current = growth[:, ids].sum(axis=1)
        candidates = []
        for stock in range(len(names)):
            if stock in ids:
                continue
            trial = ids + [stock]
            capacity = sector_capacity(sector_groups(trial))
            # Treat baskets that can reach 100% as tied on capacity.
            capped_capacity = round(min(capacity, 1.0), 8)
            # Compare daily volatility with equal starting weights.
            risk = path_stats(current + growth[:, stock])[1]
            candidates.append((-capped_capacity, risk, names[stock], stock))
        ids.append(min(candidates)[-1])
        if len(ids) >= 10:
            result = score_final(ids)
            if result is not None and (best is None or result[0] < best[0]):
                best = result

if best is None:
    raise ValueError("No feasible Client B portfolio. Review ticker coverage and sectors.")

# Test each one-stock swap using the same starting basket.
starting_ids = best[1].copy()
for pos in range(len(starting_ids)):
    for stock in range(len(names)):
        if stock in starting_ids:
            continue
        trial = starting_ids.copy()
        trial[pos] = stock
        result = score_final(trial)
        if result is not None and result[0] < best[0]:
            best = result

risk, selected_ids, weights, weighting_rule = best
selected = [names[i] for i in selected_ids]
correlations = daily_returns[selected].corr()
pair_correlations = []
for i in range(len(selected)):
    # Start at the next stock so each pair is counted once.
    for j in range(i + 1, len(selected)):
        pair_correlations.append(correlations.iloc[i, j])
mean_correlation = sum(pair_correlations) / len(pair_correlations)
print("Weighting rule:", weighting_rule)
print("Mean pairwise CAD return correlation:", round(mean_correlation, 3))
display(correlations.round(2))

Weighting rule: Inverse volatility before caps
Mean pairwise CAD return correlation: 0.073


,TD.TO,KO,T.TO,RY.TO,LMT,MO,AMZN,ABBV,AIG,TXN
TD.TO,1.00,-0.06,0.01,0.81,0.08,-0.13,0.24,0.00,0.17,0.22
KO,-0.06,1.00,0.16,-0.06,0.07,0.45,-0.06,0.37,0.23,-0.04
T.TO,0.01,0.16,1.00,-0.02,-0.10,0.10,-0.21,0.05,-0.01,-0.04
RY.TO,0.81,-0.06,-0.02,1.00,0.08,-0.09,0.21,0.02,0.13,0.16
LMT,0.08,0.07,-0.10,0.08,1.00,0.05,-0.04,0.13,0.04,-0.06
MO,-0.13,0.45,0.10,-0.09,0.05,1.00,-0.10,0.29,0.26,-0.10
AMZN,0.24,-0.06,-0.21,0.21,-0.04,-0.10,1.00,-0.17,-0.01,0.14
ABBV,0.00,0.37,0.05,0.02,0.13,0.29,-0.17,1.00,0.15,-0.02
AIG,0.17,0.23,-0.01,0.13,0.04,0.26,-0.01,0.15,1.00,-0.05
TXN,0.22,-0.04,-0.04,0.16,-0.06,-0.10,0.14,-0.02,-0.05,1.00


### Client B — Holy Grail diversification

**Why these stocks?** Our client wants a steady portfolio in Canadian dollars. We use Ray Dalio’s Holy Grail idea: hold stocks that do not all move together. We try three starting stocks with low daily volatility. At each step we test every remaining stock with the current basket on the same dates. We give priority to having enough room under the sector and stock limits. We then choose the addition with the lowest combined volatility at equal initial weights. This measure reflects both the stock’s own risk and how it moves with our holdings.

**How much do we invest?** For each feasible basket of 10–20 stocks, we test equal starting amounts and inverse-volatility weights. The second rule gives less money to more volatile stocks. We cap sectors and individual positions, keep the lowest historical volatility we find and test one round of single-stock replacements. This is a simple search, so another combination could do better.

**What evidence do we use?** From October 1, 2025 to September 30, 2026, the average
correlation between pairs of stocks was 0.073. Both portfolios held the same 10 stocks
and kept the number of shares fixed. Our starting weights gave Client B lower daily
CAD volatility (0.609%) than starting with 10% in each stock (0.653%). Our portfolio
also returned 17.015% over the period, compared with 15.642% for equal weights.

**What could go wrong?** We choose and compare weights on the same historical data. This can make the result look better than it will be next week. Even stocks from different sectors may fall together. USD holdings also expose the client to changes in the exchange rate.

## Equal-weight comparisons

For **each client**, create an equal-weighted comparison portfolio holding **exactly the same stocks** as that client’s chosen portfolio, with starting weight `1/n` per stock. Over the same clearly identified historical period, calculate **historical return** and **volatility** for the chosen portfolio and its equal-weighted counterpart. Use consistent CAD valuations and weighting assumptions.

Display one concise comparison table per client, and interpret the results in the financial justifications above. No additional graph or out-of-sample test is required.

In [15]:
rows = []
for label, allocation in [("Holy Grail", weights), ("Same stocks, equal weight", np.ones(len(selected)) / len(selected))]:
    value = (growth[:, selected_ids] * allocation).sum(axis=1)
    total_return, daily_std = path_stats(value)
    rows.append([label, total_return, daily_std])
comparison_B = pd.DataFrame(rows, columns=["Portfolio", "Historical return", "Daily CAD volatility"]).set_index("Portfolio")
print("Fixed shares:", prices_cad.index[0].date(), "to", prices_cad.index[-1].date())
comparison_pct = (comparison_B * 100).round(3)
comparison_pct.columns = ["Historical return (%)", "Daily CAD volatility (%)"]
display(comparison_pct)
difference = comparison_B.iloc[0, 1] - comparison_B.iloc[1, 1]
print(f"The strategy's daily volatility minus equal weight was {100 * difference:+.4f} percentage points.")
print("Both portfolios use the same stocks. This difference measures the effect of the starting weights.")

Fixed shares: 2025-10-01 to 2026-09-30


,Historical return (%),Daily CAD volatility (%)
Portfolio,,
Holy Grail,17.015,0.609
"Same stocks, equal weight",15.642,0.653


The strategy's daily volatility minus equal weight was -0.0442 percentage points.
Both portfolios use the same stocks. This difference measures the effect of the starting weights.


## Programmatic validation

Use Python (`assert` statements or a simple function are fine) to display whether **both** final portfolios pass each of the following checks:

- Selected tickers are valid and eligible.
- Number of stocks is between 10 and 20.
- No individual stock weight exceeds 15%.
- No single sector has more than 40% of portfolio value.
- Portfolio weights sum to 100%.
- Total stock purchases plus transaction fees do not exceed the applicable client budget.

Your code should run with the TAs’ ticker file without manually changing individual stock selections.

In [18]:
# Solve for the largest investment that fits the budget after per-stock fees.
prices = eligible_stocks.loc[selected, "Close_CAD"].to_numpy()
def cost(investment):
    shares = investment * weights / prices
    return float(np.dot(shares, prices)) + sum(calc_fee_cad(q) for q in shares)# the total cost, including the transaction fee.

low, high = 0.0, budget# try to spend all the budget.
for step in range(55):
    middle = (low + high) / 2
    if cost(middle) <= budget - 0.000001:
        low = middle
    else:
        high = middle
shares = low * weights / prices
values = shares * prices
holdings_B = pd.DataFrame({
    "Ticker": selected, "Price": eligible_stocks.loc[selected, "Close"].to_numpy(),
    "Currency": eligible_stocks.loc[selected, "Currency"].to_numpy(),
    "Shares": shares, "Value": values, "Weight": values / values.sum()
}, index=range(1, len(selected) + 1))
fees = sum(calc_fee_cad(q) for q in shares)
spent = values.sum() + fees
sector_weights = holdings_B.groupby(holdings_B["Ticker"].map(eligible_stocks["Sector"]))["Weight"].sum()
checks = {
    "Tickers are eligible and unique": set(selected).issubset(eligible_stocks.index) and holdings_B.Ticker.is_unique,
    "10 to 20 stocks": 10 <= len(selected) <= 20,
    "Long only, at most 15% per stock": (holdings_B.Weight > 0).all() and holdings_B.Weight.max() <= 0.15 + 1e-10,
    "At most 40% per sector": sector_weights.max() <= 0.40 + 1e-10,
    "Weights sum to 100%": np.isclose(holdings_B.Weight.sum(), 1),
    "Purchases plus fees fit the budget": spent <= budget and budget - spent < 0.01
}
display(pd.Series(checks, name="Pass"))
assert all(checks.values()), "Review failed checks before exporting."
sector_pct = (sector_weights * 100).round(2)
sector_pct.index.name = "Sector"
display(sector_pct.to_frame("Sector weight (%)"))

Tickers are eligible and unique       True
10 to 20 stocks                       True
Long only, at most 15% per stock      True
At most 40% per sector                True
Weights sum to 100%                   True
Purchases plus fees fit the budget    True
Name: Pass, dtype: bool

,Sector weight (%)
Sector,
Communication Services,10.09
Consumer Cyclical,6.88
Consumer Defensive,20.77
Financial Services,39.54
Healthcare,9.00
Industrials,8.22
Technology,5.50


## Final portfolio outputs

Display `Portfolio_Final_A` and `Portfolio_Final_B`, along with the budget reconciliation (purchases plus fees) for each. Generate `Stocks_Group_XX_A.csv` and `Stocks_Group_XX_B.csv` (replace `XX` with your team number). Each CSV must contain **only** the columns `Ticker` and `Shares`. The notebook must **generate** these CSV files when run, but you submit **only the notebook** to Dropbox.

In [20]:
print(f"Price date: {price_date.date()} | Purchases: CAD {values.sum():,.2f} | Fees: CAD {fees:,.2f} | Cash: CAD {budget-spent:,.2f}")
Portfolio_Final_B = holdings_B
if not official:
    print("Practice run. The table and CSV use the price date shown above.")
display(Portfolio_Final_B)
Portfolio_Final_B[["Ticker", "Shares"]].to_csv(
    f"Stocks_Group_{TEAM}_B.csv", index=False
)

Price date: 2026-09-30 | Purchases: CAD 749,991.69 | Fees: CAD 8.31 | Cash: CAD 0.00
Practice run. The table and CSV use the price date shown above.


,Ticker,Price,Currency,Shares,Value,Weight
1,TD.TO,167.330002,CAD,654.349849,109492.361455,0.145991
2,KO,86.080002,USD,735.786997,89882.790569,0.119845
3,T.TO,11.950000,CAD,6330.514462,75649.646615,0.100867
4,RY.TO,279.940002,CAD,401.867374,112498.753782,0.150000
5,LMT,509.250000,USD,85.271794,61625.237989,0.082168
6,MO,67.339996,USD,689.339863,65876.217225,0.087836
7,AMZN,249.149994,USD,146.034467,51634.315211,0.068847
8,ABBV,261.589996,USD,181.806664,67492.124734,0.089990
9,AIG,74.419998,USD,706.091958,74571.536467,0.099430
10,TXN,280.089996,USD,103.824800,41268.707835,0.055026


## Contribution and AI Declaration

**The following team members made a meaningful contribution to this assignment:**

Enter the names of all contributing members below (one name per line):

**To the best of our knowledge, no generative AI was used in preparing this submission.**

List only members who made a meaningful contribution. Contact the instructor before the deadline if someone should be omitted or if the team cannot make the declaration.